Training and validation were performed with the SwissProt Training/Validation dataset used to train the DeepLoc 2.0 server. Testing was performed using the DeepLoc 2.0 Human Protein Atlas (HPA) test set. The data is available at: https://services.healthtech.dtu.dk/services/DeepLoc-2.0/4-Data.php

**References**

Thumuluri, V., Almagro Armenteros, J. J., Johansen, A. R., Nielsen, H. & Winther, O. (2022). *DeepLoc 2.0: multi-label subcellular localization prediction using protein language models.* Nucleic Acids Research, 50(W1), W228–W234. https://doi.org/10.1093/nar/gkac278

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

In [2]:
df = pd.read_csv("Swissprot_Train_Validation_dataset.csv")
df.head()
df.shape

(28303, 16)

In [3]:
df = df.dropna(subset = ["Sequence", "Nucleus"])
df.shape

(28303, 16)

In [4]:
df = df[["Kingdom", "Partition", "Nucleus", "Sequence"]]
df.head()

,Kingdom,Partition,Nucleus,Sequence
0,Metazoa,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
1,Metazoa,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...
2,Metazoa,3,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...
3,Metazoa,1,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...
4,Metazoa,1,1.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...


In [5]:
df_test = pd.read_csv("hpa_testset.csv")
df_test.head()

,sid,Cell membrane,Cytoplasm,Endoplasmic reticulum,Golgi apparatus,Lysosome/Vacuole,Mitochondrion,Nucleus,Peroxisome,Lengths,fasta
0,ENSP00000355743,0,0,0,0,0,0,1,0,496,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...
1,ENSP00000429628,0,0,0,0,0,0,1,0,173,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...
2,ENSP00000266544,0,0,0,0,0,1,0,0,377,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...
3,ENSP00000470652,0,0,0,1,0,0,0,0,410,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...
4,ENSP00000460751,0,0,0,0,0,0,1,0,163,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...


In [6]:
df_test.rename(columns={"fasta": "Sequence"}, inplace=True)
df_test = df_test.dropna(subset = ["Sequence", "Nucleus"])
df_test.shape

(1717, 11)

In [7]:
df_test = df_test[["Nucleus", "Sequence"]]
df_test.head()

,Nucleus,Sequence
0,1,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...
1,1,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...
2,0,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...
3,0,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...
4,1,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...


In [8]:
#feature engineering - dipeptide composition (as a fraction of number of dipeptides, which is = (sequence length) - 1 )
#dipeptide composition includes some positional information: which amino acids often occur adjacent to each other

amino_acids = "ACDEFGHIKLMNPQRSTVWY"

aa_to_idx = {aa: i for i, aa in enumerate(amino_acids)}

def dipeptide_comp(sequence):
    # Convert valid amino acids to indices
    indices = np.array([aa_to_idx.get(aa, -1) for aa in sequence])

    first = indices[:-1]
    second = indices[1:]

    # Keep only pairs residues are standard amino acids (not X, B, U)
    valid = (first >= 0) & (second >= 0)

    first = first[valid]
    second = second[valid]

    # Encode each dipeptide as an integer 0–399
    dipeptide_indices = first * 20 + second

    counts = np.bincount(dipeptide_indices, minlength=400)

    # Normalise by number of valid dipeptides
    n_valid = len(dipeptide_indices)

    if n_valid == 0:
        return np.zeros(400)

    return counts / n_valid

In [9]:
df["input_vectors"] = df["Sequence"].apply(dipeptide_comp)
df.head()

,Kingdom,Partition,Nucleus,Sequence,input_vectors
0,Metazoa,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...,"[0.03278688524590164, 0.0, 0.00327868852459016..."
1,Metazoa,4,1.0,MAAAAAAAAAAGAAGGRGSGPGRRRHLVPGAGGEAGEGAPGGAGDY...,"[0.03278688524590164, 0.0, 0.00327868852459016..."
2,Metazoa,3,0.0,MAAAAAAAAALGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...,"[0.022105263157894735, 0.002105263157894737, 0..."
3,Metazoa,1,0.0,MAAAAAAAAATEQQGSNGPVKKSMREKAVERRNVNKEHNSNFKAGY...,"[0.025078369905956112, 0.0, 0.0031347962382445..."
4,Metazoa,1,1.0,MAAAAAAAAATNGTGGSSGMEVDAAVVPSVMACGVTGSVSVALHPL...,"[0.027607361963190184, 0.003067484662576687, 0..."


In [10]:
df_test["input_vectors"] = df_test["Sequence"].apply(dipeptide_comp)
df_test.head()

,Nucleus,Sequence,input_vectors
0,1,MAAAAGPGAALSPRPCDSDPATPGAQSPKDDNEDNSNDGTQPSKRR...,"[0.014141414141414142, 0.0, 0.0, 0.00808080808..."
1,1,MAAAALRDPAQVPVAADLLTDHEEGYVTFEDVAVYFSQEEWRLLDD...,"[0.023255813953488372, 0.0, 0.0058139534883720..."
2,0,MAAAAQSRVVRVLSMSRSAITAIATSVCHGPPCRQLHHALMPHGKG...,"[0.007978723404255319, 0.0, 0.0, 0.00265957446..."
3,0,MAAAAVGAGHGAGGPGAASSSGGAREGARVAALCLLWYALSAGGNV...,"[0.014669926650366748, 0.0, 0.0024449877750611..."
4,1,MAAAAVTRGTPGENSHHLKIFLPKKLLECLPRCPLLPPERLRWNTN...,"[0.024691358024691357, 0.006172839506172839, 0..."


In [11]:
#logistic regression model with 5-fold cross-validation across homology-partitioned training datasets
#first fold of 5-fold cross-validation will use homology partition 0 as the validation set
train_df = df[df["Partition"].isin([1, 2, 3, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 0]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_test = np.stack(df_test["input_vectors"].values)
y_test = np.stack(df_test["Nucleus"].values)

In [12]:
#feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

In [13]:
#initialise the class
model = LogisticRegression()
#train the model
model.fit(X_train_scaled, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [14]:
yhat = model.predict(X_cv_scaled)

#extract probability of output label 1
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
print(y_prob[0:200])

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

[0.77196851 0.33477287 0.25498555 0.09194775 0.17837023 0.23553109
 0.15942585 0.11955129 0.39286388 0.57409329 0.69192298 0.60575869
 0.14711523 0.10997375 0.12302329 0.47731029 0.57123503 0.06653807
 0.10270285 0.05750032 0.0701054  0.16709966 0.60927634 0.73014489
 0.20992595 0.22978459 0.41752845 0.76327816 0.77103563 0.28218502
 0.14997051 0.23293513 0.00375883 0.35575415 0.36184945 0.37300008
 0.11311562 0.50030253 0.14578263 0.31015392 0.41830874 0.44130873
 0.35444517 0.81422133 0.36676789 0.59455125 0.10213341 0.41695273
 0.74350301 0.42141424 0.24257089 0.29880252 0.05765937 0.12176431
 0.76305972 0.80425777 0.09437888 0.0329152  0.1155016  0.07074434
 0.36602311 0.12842056 0.26769883 0.30635355 0.20775402 0.19218125
 0.3180794  0.44625545 0.47229952 0.32052146 0.78714842 0.68118717
 0.70542762 0.70946314 0.86115027 0.79190007 0.27468362 0.41625237
 0.71199218 0.77647803 0.72889723 0.35872493 0.33221377 0.92704794
 0.9104926  0.11019146 0.23226553 0.003112   0.35435545 0.5279

In [15]:
#lower the threshold to 0.4
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.4
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7273184638604729
              precision    recall  f1-score   support

         0.0       0.82      0.77      0.79      4092
         1.0       0.56      0.64      0.59      1871

    accuracy                           0.73      5963
   macro avg       0.69      0.70      0.69      5963
weighted avg       0.74      0.73      0.73      5963

[[3148  944]
 [ 682 1189]]


In [16]:
#lower the threshold to 0.3
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.3
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6828777461009559
              precision    recall  f1-score   support

         0.0       0.86      0.64      0.73      4092
         1.0       0.50      0.78      0.61      1871

    accuracy                           0.68      5963
   macro avg       0.68      0.71      0.67      5963
weighted avg       0.75      0.68      0.69      5963

[[2614 1478]
 [ 413 1458]]


In [17]:
#2nd fold of 5-fold cross-validation will use homology partition 1 as the validation set
train_df = df[df["Partition"].isin([0, 2, 3, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 1]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

#initialise the class
model = LogisticRegression()
#train the model
model.fit(X_train_scaled, y_train)

yhat = model.predict(X_cv_scaled)

#extract probability of output label 1
y_prob = model.predict_proba(X_cv_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7138139790864062
              precision    recall  f1-score   support

         0.0       0.75      0.84      0.79      3536
         1.0       0.62      0.49      0.54      1915

    accuracy                           0.71      5451
   macro avg       0.68      0.66      0.67      5451
weighted avg       0.70      0.71      0.70      5451

[[2960  576]
 [ 984  931]]


In [18]:
#lower the threshold to 0.4
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.40
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6973032471106219
              precision    recall  f1-score   support

         0.0       0.78      0.74      0.76      3536
         1.0       0.56      0.62      0.59      1915

    accuracy                           0.70      5451
   macro avg       0.67      0.68      0.68      5451
weighted avg       0.71      0.70      0.70      5451

[[2606  930]
 [ 720 1195]]


In [19]:
#lower the threshold to 0.3
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.3
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6584113006787745
              precision    recall  f1-score   support

         0.0       0.82      0.60      0.70      3536
         1.0       0.51      0.76      0.61      1915

    accuracy                           0.66      5451
   macro avg       0.67      0.68      0.65      5451
weighted avg       0.71      0.66      0.67      5451

[[2138 1398]
 [ 464 1451]]


In [20]:
#3rd fold of 5-fold cross-validation will use homology partition 2 as the validation set
train_df = df[df["Partition"].isin([0, 1, 3, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 2]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

#initialise the class
model = LogisticRegression()
#train the model
model.fit(X_train_scaled, y_train)

yhat = model.predict(X_cv_scaled)

#extract probability of output label 1
y_prob = model.predict_proba(X_cv_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7445471994416332
              precision    recall  f1-score   support

         0.0       0.78      0.86      0.82      3800
         1.0       0.65      0.52      0.58      1931

    accuracy                           0.74      5731
   macro avg       0.72      0.69      0.70      5731
weighted avg       0.74      0.74      0.74      5731

[[3264  536]
 [ 928 1003]]


In [21]:
#lower the threshold to 0.4
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.40
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7352992496946432
              precision    recall  f1-score   support

         0.0       0.82      0.77      0.79      3800
         1.0       0.60      0.66      0.63      1931

    accuracy                           0.74      5731
   macro avg       0.71      0.72      0.71      5731
weighted avg       0.74      0.74      0.74      5731

[[2938  862]
 [ 655 1276]]


In [22]:
#lower the threshold to 0.3
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.3
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6918513348455767
              precision    recall  f1-score   support

         0.0       0.86      0.64      0.73      3800
         1.0       0.53      0.79      0.63      1931

    accuracy                           0.69      5731
   macro avg       0.69      0.72      0.68      5731
weighted avg       0.75      0.69      0.70      5731

[[2447 1353]
 [ 413 1518]]


In [23]:
#4th fold of 5-fold cross-validation will use homology partition 3 as the validation set
train_df = df[df["Partition"].isin([0, 1, 2, 4])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 3]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

#initialise the class
model = LogisticRegression()
#train the model
model.fit(X_train_scaled, y_train)

yhat = model.predict(X_cv_scaled)

#extract probability of output label 1
y_prob = model.predict_proba(X_cv_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7205056179775281
              precision    recall  f1-score   support

         0.0       0.75      0.85      0.80      3710
         1.0       0.63      0.49      0.55      1986

    accuracy                           0.72      5696
   macro avg       0.69      0.67      0.67      5696
weighted avg       0.71      0.72      0.71      5696

[[3138  572]
 [1020  966]]


In [24]:
#lower the threshold to 0.4
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.40
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.710498595505618
              precision    recall  f1-score   support

         0.0       0.80      0.75      0.77      3710
         1.0       0.58      0.64      0.61      1986

    accuracy                           0.71      5696
   macro avg       0.69      0.69      0.69      5696
weighted avg       0.72      0.71      0.71      5696

[[2775  935]
 [ 714 1272]]


In [25]:
#lower the threshold to 0.3
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.3
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6753862359550562
              precision    recall  f1-score   support

         0.0       0.84      0.62      0.71      3710
         1.0       0.52      0.78      0.62      1986

    accuracy                           0.68      5696
   macro avg       0.68      0.70      0.67      5696
weighted avg       0.73      0.68      0.68      5696

[[2307 1403]
 [ 446 1540]]


In [26]:
#5th fold of 5-fold cross-validation will use homology partition 4 as the validation set
train_df = df[df["Partition"].isin([0, 1, 2, 3])]
X_train = np.stack(train_df["input_vectors"].values)
y_train = np.stack(train_df["Nucleus"].values)

cv_df = df[df["Partition"] == 4]
X_cv = np.stack(cv_df["input_vectors"].values)
y_cv = np.stack(cv_df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_cv_scaled = scaler.transform(X_cv)

#initialise the class
model = LogisticRegression()
#train the model
model.fit(X_train_scaled, y_train)

yhat = model.predict(X_cv_scaled)

#extract probability of output label 1
y_prob = model.predict_proba(X_cv_scaled)[:, 1]

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7233614060783595
              precision    recall  f1-score   support

         0.0       0.74      0.87      0.80      3445
         1.0       0.68      0.48      0.56      2017

    accuracy                           0.72      5462
   macro avg       0.71      0.67      0.68      5462
weighted avg       0.72      0.72      0.71      5462

[[2984  461]
 [1050  967]]


In [27]:
#lower the threshold to 0.4
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.40
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.7277554009520322
              precision    recall  f1-score   support

         0.0       0.79      0.78      0.78      3445
         1.0       0.63      0.64      0.63      2017

    accuracy                           0.73      5462
   macro avg       0.71      0.71      0.71      5462
weighted avg       0.73      0.73      0.73      5462

[[2693  752]
 [ 735 1282]]


In [28]:
#lower the threshold to 0.3
y_prob = model.predict_proba(X_cv_scaled)[:, 1]
threshold = 0.3
yhat = (y_prob >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_cv, yhat))
print(classification_report(y_cv, yhat))
print(confusion_matrix(y_cv, yhat))

Accuracy: 0.6942511900402782
              precision    recall  f1-score   support

         0.0       0.83      0.64      0.73      3445
         1.0       0.56      0.78      0.65      2017

    accuracy                           0.69      5462
   macro avg       0.70      0.71      0.69      5462
weighted avg       0.73      0.69      0.70      5462

[[2220 1225]
 [ 445 1572]]


In [29]:
#C value = inverse of regularisation strength in scikit-learn
C_values = [0.01, 0.1, 1, 10, 100]
accuracies = []
f1_scores = []

#cross-validation for different C-values using fold 5
for c in C_values:
    model = LogisticRegression(C=c)
    model.fit(X_train_scaled, y_train)
    y_prob = model.predict_proba(X_cv_scaled)[:, 1]
    threshold = 0.40
    yhat = (y_prob >= threshold).astype(int)
    accuracy = accuracy_score(y_cv, yhat)
    accuracies.append(accuracy)
    print(f"C={c}, Accuracy: {accuracy:.3f}")
    f1 = f1_score(y_cv, yhat)
    f1_scores.append(f1)
    print(f"C={c}, f1: {f1:.3f}")

C=0.01, Accuracy: 0.727
C=0.01, f1: 0.631
C=0.1, Accuracy: 0.726
C=0.1, f1: 0.630
C=1, Accuracy: 0.728
C=1, f1: 0.633
C=10, Accuracy: 0.729
C=10, f1: 0.634
C=100, Accuracy: 0.729
C=100, f1: 0.634


In [30]:
#final model
X_train = np.stack(df["input_vectors"].values)
y_train = np.stack(df["Nucleus"].values)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(C = 10)
model.fit(X_train_scaled, y_train)

y_prob = model.predict_proba(X_test_scaled)[:, 1]
threshold = 0.4
yhat = (y_prob >= threshold).astype(int)

y_prob = model.predict_proba(X_train_scaled)[:, 1]
threshold = 0.4
yhat_train = (y_prob >= threshold).astype(int)

print("Accuracy (training set):", accuracy_score(y_train, yhat_train))
print("F1 score (training set):", f1_score(y_train, yhat_train))
print("Accuracy (test set):", accuracy_score(y_test, yhat))
print("F1 score (test set):", f1_score(y_test, yhat))
print(classification_report(y_test, yhat))
print(confusion_matrix(y_test, yhat))

Accuracy (training set): 0.7347984312617037
F1 score (training set): 0.6309371619628282
Accuracy (test set): 0.6115317414094351
F1 score (test set): 0.5989176187612748
              precision    recall  f1-score   support

           0       0.58      0.67      0.62       824
           1       0.65      0.56      0.60       893

    accuracy                           0.61      1717
   macro avg       0.61      0.61      0.61      1717
weighted avg       0.62      0.61      0.61      1717

[[552 272]
 [395 498]]
